In [1]:
from pyspark.sql import SparkSession

# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("MAST30034 Tutorial 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/19 12:36:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/19 12:36:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
import pandas as pd

# data output directory is
output_relative_dir = '../../data/'
abs_output_dir = output_relative_dir + 'raw_abs'

# Reading in the POA <-> SA2 mapping dataset produced in the previous run 
poa_to_sa2 = pd.read_parquet(f"{abs_output_dir}/poa_to_sa2.parquet")
print(poa_to_sa2.info())
print(poa_to_sa2.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2644 entries, 0 to 2643
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   POA_CODE_2021  2644 non-null   object 
 1   SA2_CODE_2021  2644 non-null   object 
 2   mb_count       2644 non-null   int64  
 3   poa_total      2644 non-null   int64  
 4   ratio          2644 non-null   float64
dtypes: float64(1), int64(2), object(2)
memory usage: 103.4+ KB
None
  POA_CODE_2021 SA2_CODE_2021  mb_count  poa_total     ratio
0          0800     701011002        93         93  1.000000
1          0810     701021025        69        459  0.150327
2          0812     701021022        74        266  0.278195
3          0820     701011008        78        365  0.213699
4          0822     702041063       131        511  0.256360


In [3]:
poa_to_sa2['POA_CODE_2021'].duplicated().sum()

np.int64(0)

In [4]:
# Reading in the external dataset with the business indicators of SEIFA and Income datasets produced in the previous run 
features_df = pd.read_parquet(f"{abs_output_dir}/loaded_features_df.parquet")
print(features_df.info())
print(features_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2472 entries, 0 to 2471
Data columns (total 7 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   SA2_CODE_2021                   2472 non-null   object 
 1   SA2_NAME_2021                   2366 non-null   object 
 2   IRSAD_score                     2353 non-null   float64
 3   IRSAD_decile                    2353 non-null   float64
 4   median_personal_income_weekly   2472 non-null   int64  
 5   median_family_income_weekly     2472 non-null   int64  
 6   median_household_income_weekly  2472 non-null   int64  
dtypes: float64(2), int64(3), object(2)
memory usage: 135.3+ KB
None
  SA2_CODE_2021                    SA2_NAME_2021  IRSAD_score  IRSAD_decile  \
0     101021007                        Braidwood       1001.0           6.0   
1     101021008                          Karabar        982.0           5.0   
2     101021009                       Q

In [11]:
# Reading in the actual tbl consumer data 
tbl_consumer_df = pd.read_csv("../../data/tables/part_1/tbl_consumer.csv", sep="|")
print(tbl_consumer_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 499999 entries, 0 to 499998
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   name         499999 non-null  object
 1   address      499999 non-null  object
 2   state        499999 non-null  object
 3   postcode     499999 non-null  int64 
 4   gender       499999 non-null  object
 5   consumer_id  499999 non-null  int64 
dtypes: int64(2), object(4)
memory usage: 22.9+ MB
None


Since the "postcode" column in tbl_consumer_df is stored as an integer and the poa_to_sa2 stores "POA_CODE_2021" as a string of four characters, we need to make sure their data typesa are consistent. Therefore, postcode columns data type will be adjusted accordingly. 

In [12]:
tbl_consumer_df['postcode'] = tbl_consumer_df['postcode'].astype(str).str.zfill(4)

In [13]:
# Merging consumer data to the SA2 data via postcode
consumer_sa2_df = tbl_consumer_df.merge(
    poa_to_sa2, 
    left_on="postcode", 
    right_on="POA_CODE_2021", 
    how= "left" )
print(consumer_sa2_df.info)
print(consumer_sa2_df.shape)


<bound method DataFrame.info of                      name                          address state postcode  \
0        Yolanda Williams       413 Haney Gardens Apt. 742    WA     6935   
1              Mary Smith                  3764 Amber Oval   NSW     2782   
2           Jill Jones MD               40693 Henry Greens    NT     0862   
3         Lindsay Jimenez        00653 Davenport Crossroad   NSW     2780   
4       Rebecca Blanchard    9271 Michael Manors Suite 651    WA     6355   
...                   ...                              ...   ...      ...   
499994      Jessica Avila    508 Miranda Overpass Apt. 218   QLD     4400   
499995    Steven Thornton  7913 Schwartz Mission Suite 483   VIC     3097   
499996      Christy Smith   5681 Zachary Mountain Apt. 060   NSW     2756   
499997       Donna Sutton                54140 Jacob Point   VIC     3989   
499998     Hannah Wilkins                61055 Long Valley   NSW     1755   

             gender  consumer_id POA_CODE_2

In [14]:
consumer_sa2_df = consumer_sa2_df.merge(features_df, on='SA2_CODE_2021', how='left')

In [15]:
consumer_sa2_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 499999 entries, 0 to 499998
Data columns (total 17 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   name                            499999 non-null  object 
 1   address                         499999 non-null  object 
 2   state                           499999 non-null  object 
 3   postcode                        499999 non-null  object 
 4   gender                          499999 non-null  object 
 5   consumer_id                     499999 non-null  int64  
 6   POA_CODE_2021                   416818 non-null  object 
 7   SA2_CODE_2021                   416818 non-null  object 
 8   mb_count                        416818 non-null  float64
 9   poa_total                       416818 non-null  float64
 10  ratio                           416818 non-null  float64
 11  SA2_NAME_2021                   415735 non-null  object 
 12  IRSAD_score     

In [16]:
# Checking for missing SA2 values where it is NaN
consumer_sa2_df['SA2_CODE_2021'].isna().sum()

np.int64(83181)

In [17]:
# Identifying the reason for the NaN SA2 values that might have occured during the merging due to missing matching postcodes 
consumer_postcodes = set(tbl_consumer_df['postcode'].unique())
poa_postcodes = set(poa_to_sa2['POA_CODE_2021'].unique())

missing_postcodes = consumer_postcodes - poa_postcodes
len(missing_postcodes)
list(missing_postcodes)[:20]

['4475',
 '1296',
 '6711',
 '5942',
 '3989',
 '1035',
 '1891',
 '2123',
 '2314',
 '1119',
 '4801',
 '6957',
 '1485',
 '1831',
 '5001',
 '2608',
 '1154',
 '1181',
 '1138',
 '6435']

Inspecting the missing SA2 values during the merge shows 83,181 rows with NaN SA2 value, this is around 16.6% (83,181/499,999 of total). After closely investigating and researching, it was found that these postcodes are reserved for Australia's non geographic post code ranges exclusively. The ABS Postal Area is built from mesh blocks but since these codes do not have a geographic location linekd to it, there is no SA2 to allocate it to. 

In [18]:
# Building a dataset with SEIFA and income results averaged per state and named accordingly 

# In post codes and SA2 codes, the first character of the code, is often associated to its relevant state in ABS data
STATE_TO_CODE = {
    '1': 'NSW', '2': 'VIC', '3': 'QLD', '4': 'SA',
    '5': 'WA', '6': 'TAS', '7': 'NT', '8': 'ACT', '9': 'OT',
}

feature_cols = ['IRSAD_score', 'IRSAD_decile',
                          'median_personal_income_weekly', 'median_family_income_weekly', 'median_household_income_weekly']

# Consider the first character of the SA2_CODE_2021 and map it to its relevant state and rename that column as "state"
state_avg = features_df['SA2_CODE_2021'].str[0].map(STATE_TO_CODE).rename('state')
# Group by the states and get the average 
state_averages = features_df.groupby(state_avg)[feature_cols].mean().reset_index()
state_averages

,state,IRSAD_score,IRSAD_decile,median_personal_income_weekly,median_family_income_weekly,median_household_income_weekly
0,ACT,1091.709091,8.709091,1107.367647,2669.720588,2281.397059
1,NSW,1011.653355,5.817891,841.284161,2207.085404,1844.251553
2,NT,944.629032,4.822581,881.385714,1861.257143,1818.557143
3,OT,934.500000,3.000000,549.166667,1275.666667,1272.833333
4,QLD,980.555347,4.945591,807.390511,2024.947080,1707.624088
5,SA,963.415663,4.373494,739.528409,1787.261364,1422.136364
6,TAS,939.885417,3.479167,710.079208,1631.267327,1310.475248
7,VIC,1008.920078,5.884990,823.540076,2143.929389,1760.536260
8,WA,995.205761,5.415638,864.846442,2151.985019,1746.288390


In [22]:
# Fill missing SEIFA and income feature values using each consumer's state average
# (used for consumers whose SA2 was never found, or whose SA2 has no SEIFA data)
state_avg_lookup = state_averages.set_index('state')

# consumer_sa2_df['approx_sa2'] = consumer_sa2_df['SA2_CODE_2021'].isna()

for col in feature_cols:
    # Records True or False for the rows of each column
    # True = this value is currently missing and is about to be filled with a state average below
    consumer_sa2_df[f'{col}_is_approximated'] = consumer_sa2_df[col].isna()

    # For each feature, map their state's average value for this feature
    mapping = consumer_sa2_df['state'].map(state_avg_lookup[col])

    # If for that column, the row entry is missing, fill it with the state average
    consumer_sa2_df[col] = consumer_sa2_df[col].fillna(mapping)

In [23]:
consumer_sa2_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 499999 entries, 0 to 499998
Data columns (total 22 columns):
 #   Column                                          Non-Null Count   Dtype  
---  ------                                          --------------   -----  
 0   name                                            499999 non-null  object 
 1   address                                         499999 non-null  object 
 2   state                                           499999 non-null  object 
 3   postcode                                        499999 non-null  object 
 4   gender                                          499999 non-null  object 
 5   consumer_id                                     499999 non-null  int64  
 6   POA_CODE_2021                                   416818 non-null  object 
 7   SA2_CODE_2021                                   416818 non-null  object 
 8   mb_count                                        416818 non-null  float64
 9   poa_total                 

In [ ]:
# data output directory is
output_relative_dir = '../../data/'
abs_output_dir = output_relative_dir + 'raw_abs'

# save the cleaned POA <-> SA2 mapping df to be reused in the next step 
consumer_sa2_df.to_parquet(f"{abs_output_dir}/consumer_sa2.parquet", index=False)
